## 2. Filtering, extracting and computing basic metrics on buildings from S2 Partitions.
This notebook is divided into to main parts. First, it gets rid of the buildings that are not within the boundaries of the selected region. Normally, this geospatial operation is done using a clip, however, with complex polygons and larger datasets it increases the computational time. This notebook, breakes down the problem to smaller tiles and it checks the location of the tile in relation to the boundaries. The final output is similar to the following image, on the right it shows the boundaries of the selected area, which is a large administrative division, on the right, the output after filtering out buildings. Notice the scale in kilometers, the total area is 700km2 and the number of buildings goes from 3 million to 200,000. In this first part the **Latitude** and **Longitude** are also computed and added as final attributes.

In the second part, it computes the **number of facades**, **perimeter** and **area**.

### Initial configuration

    config_str = """{"COS_ENDPOINT_URL": "https://s3.direct.eu-de.cloud-object-storage.appdomain.cloud",
    "COUNTRY_NAME": "India",
    "VIDA_PARQUET_BUCKET": "parquets",
    "VIDA_S2_PARTITIONS": "onboarding-bucket-j",
    "COS_AUTH_ENDPOINT_URL":"https://iam.cloud.ibm.com/oidc/token",
    "COS_APIKEY": "xxx",
    "UTILS_BUCKET": "notebook-utils-bucket",
    "VIDA_COUNTRIES_BUILDINGS": "onboarding-bucket-j"}"""

* `COS_ENDPOINT_URL`: The S3 API endpoint for your IBM COS instance.
* `COUNTRY_NAME`: Defines the target country to filter the google dataset.
* `VIDA_PARQUET_BUCKET`: The name of the source bucket where the Parquet files are stored.
* `COS_AUTH_ENDPOINT_URL`: The authentication endpoint for generating IBM IAM tokens.
* `COS_APIKEY`: Your unique IBM Cloud API key for programmatic access.
* `UTILS_BUCKET`: Where the .py file to download the package `filtering_grid_generator`.
* `VIDA_COUNTRIES_BUILDINGS`: Bucket to upload results.

In [1]:
from datetime import datetime
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-10-02 08:18:40'

In [2]:
# Import necessary libraries
import pyarrow.parquet as pq
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely
import time
import threading
import datetime 
import jaydebeapi as jdbc
import jpype
import os
from tqdm import tqdm
from botocore.client import Config
import ibm_boto3
import io
import time
import pickle

from pyproj import Geod

geod = Geod(ellps="WGS84")

In [4]:
# Read notebook configuration
import getpass
import json

#config_str = getpass.getpass('Enter your prepared config: ')
config = json.loads(config_str)


The main functions to compute perimeter, area and number of facades are defined in the following cell.

* **get_inner_faces(interiors)** iterates through each hole (interior) and counts the number of coordinate points that define its boundary.
* **get_inner_perimeter(interiors)** uses the `Geod` library to perform a geodesic calculation, then it sums the perimeters of all holes.
* **calculations** applies the two defined functions and also computes the area for each polygon. All of these attributes are added to the main dataset.

In [5]:
def get_inner_faces(interiors):
    total = 0
    for interior in interiors:
        total += len(interior.coords) - 1
    return total

def get_inner_perimeter(interiors):
    geod = Geod(ellps="WGS84")
    total = 0
    for interior in interiors:
        total += abs(geod.geometry_area_perimeter(interior)[1])
    return total


def calculations(buildings_in_polygon):
    
    # filter out google buildings vith cobfidence < miniminal_vida_confidence
    google_buildings = buildings_in_polygon[(buildings_in_polygon.bf_source == 'google') & (buildings_in_polygon.confidence > miniminal_vida_confidence)]
    microsoft_buildings = buildings_in_polygon[buildings_in_polygon.bf_source == 'microsoft']
    
    buildings_in_polygon = pd.concat([google_buildings, microsoft_buildings])
    
    del google_buildings
    del microsoft_buildings
    
    buildings_in_polygon['area_in_meters'] = buildings_in_polygon["geometry"].apply(lambda g: abs(geod.geometry_area_perimeter(g)[0]))

    # Compute Perimeter
    buildings_in_polygon.insert(0,"perimeter_in_meters",0)
    buildings_in_polygon['perimeter_in_meters'] = buildings_in_polygon["geometry"].apply(lambda g: (abs(geod.geometry_area_perimeter(g)[1]) + get_inner_perimeter(g.interiors)) if (g.geom_type == "Polygon") else (abs(geod.geometry_area_perimeter(g)[1])))

    # Compute number of faces
    buildings_in_polygon.insert(1,"building_faces",0)
    buildings_in_polygon['building_faces'] = buildings_in_polygon["geometry"].apply(lambda g: (len(g.exterior.coords) - 1 + get_inner_faces(g.interiors)) if (g.geom_type == "Polygon") else 0)

    
    return buildings_in_polygon

Google dataset comes with a range of condifence. To filter out noice, a **threshold of 0.7** was decided. To change the threshold change the following cell.

In [6]:
miniminal_vida_confidence = 0.7

The next cell ask for three pats:
The first one refers to the administrative boundaries of the studied the selected area and it will be stored in **input_file**. For consistency, the geojson file should be the same use to run [the first notebook.](https://10.243.64.5/user/julian/lab/tree/handover/notebooks/1_downloading_S2grid_datasets.ipynb). The second one, where the parquet files were stored using the first notebook, under the variables name **parquets_folder**. The last one **filtered_parquets_folder** it is where the output of the main function will be stored.

In [7]:
# init S3 client in order to work with last tiff file version
cos_client = ibm_boto3.client(service_name='s3',
                                  ibm_api_key_id=config["COS_APIKEY"],
                                  ibm_auth_endpoint="https://iam.cloud.ibm.com/oidc/token",
                                  config=Config(signature_version='oauth'),
                                  endpoint_url=config["COS_ENDPOINT_URL"])


# import external utils library
response = cos_client.list_objects_v2(Bucket=config["UTILS_BUCKET"])

utils_to_download = ['regions_S2_ids.pkl', 'filtering_grid_generator.py']

try:
    for obj in response['Contents']:
        name = obj['Key']
        if name in utils_to_download:
            streaming_body_1 = cos_client.get_object(Bucket=config["UTILS_BUCKET"], Key=name)['Body']
            print("Copying to localStorage :  " + name)
            with io.FileIO(name, 'w') as file:
                for i in io.BytesIO(streaming_body_1.read()):
                    file.write(i)
                    
    from filtering_grid_generator import GridGenerator
    
    print('External utils succesfully imported')
except Exception as e:
    print('Error occured: ', e)


Copying to localStorage :  filtering_grid_generator.py
Copying to localStorage :  regions_S2_ids.pkl
External utils succesfully imported


The next cell ask for two paths. The first one "administrative boundaries" refers to the boundaries of the selected area where the script will focus to obtain the building footprints. The second one, where the parquet files are going to be locally stored.

In [8]:
input_file = "/home/julian/notebooks/handover/notebooks/district_boundaries/Patan.geojson"
output_folder = "/home/julian/notebooks/"

In [9]:
Name_region= "Patan"

### Download all the S2 partitions


In [10]:
S2_partitions_bucket_response = cos_client.list_objects_v2(Bucket=config["VIDA_S2_PARTITIONS"]).get('Contents', [])
S2_partitions_bucket_objects = [i['Key'] for i in S2_partitions_bucket_response]



parquets_folder = os.path.join(output_folder,'S2_partitions')
if not os.path.exists(parquets_folder):
    os.makedirs(parquets_folder)
    
filtered_parquets_folder = os.path.join(output_folder,'states_filteded_parquets')
if not os.path.exists(filtered_parquets_folder):
    os.makedirs(filtered_parquets_folder)
    
for f in tqdm(S2_partitions_bucket_objects, total=len(S2_partitions_bucket_objects)):
    
    try:
        cos_client.download_file(Bucket=config["VIDA_S2_PARTITIONS"],Key=f,Filename=f'{parquets_folder}/{f}')
    except Exception as e:
        print(f'While downloading exception occurred: {e}')
        try:
            
            time.sleep(5)
            cos_client.download_file(Bucket=config["VIDA_S2_PARTITIONS"],Key=f,Filename=f'{parquets_folder}/{f}')
        except Exception as e:
            print(f'While second downloading attempt exception occurred: {e}')

  0%|          | 0/58 [00:00<?, ?it/s]

While downloading exception occurred: [Errno 2] No such file or directory: '/home/julian/notebooks/S2_partitions//home/julian/notebooks/Patan_buildings.parquet.B5EeDC64'


  2%|▏         | 1/58 [00:06<06:02,  6.36s/it]

While second downloading attempt exception occurred: [Errno 2] No such file or directory: '/home/julian/notebooks/S2_partitions//home/julian/notebooks/Patan_buildings.parquet.9c198020'


100%|██████████| 58/58 [03:06<00:00,  3.22s/it]


In [11]:
geojson_read = gpd.read_file(input_file)
regions_polygons = {
    Name_region: [
        geojson_read.geometry.explode()[0]
        ]
}

In [12]:
#regions_S2_ids = pickle.load(open('regions_S2_ids.pkl','rb'))
pickle_path = os.path.join(parquets_folder, 'regions_S2_ids.pkl')
regions_S2_ids = pickle.load(open(pickle_path,'rb'))

The coming cell is the most computationally intensive part of this script. It starts by creating a grid of polygons (rectangles) over the boundaries of the selected area. The density of the grid depends on n; the higher the n, the more granular the grid will be. These grid cells are then sorted into two categories: those inside and those outside the main polygon. This allows the script to iterate through the "inside" grid cells and quickly collect all buildings contained within them. A preliminary dataset of "guaranteed" inside buildings is created. The final, precise geometric test is then performed only on the few ambiguous points located near the polygon's border.

In [13]:
grids = GridGenerator()

for region_name, region_polygons in regions_polygons.items():

    print('\033[96mProcessing region', region_name)

    region_S2_ids = regions_S2_ids[region_name]

    buildings_inside_region = []

    for pidx, region_polygon in enumerate(region_polygons):

        print('\033[93m    processing pidx', pidx)

        inside_polygon_grid, outside_polygon_grid = grids.generate_grids(region_polygon, n=150)

        buildings_inside_polygon = []

        buildings_inside_polygon_found = 0

        for s2_idx, S2_id in enumerate(region_S2_ids):

            try:
                file_name = f'India_{S2_id}.parquet'

                print(f'\033[93m        processing file_name {file_name} idx {s2_idx} of {len(region_S2_ids)}')

                df = gpd.read_parquet(os.path.join(parquets_folder, file_name))
                df['longitude'] = df['geometry'].apply(lambda g: g.centroid.xy[0][0])
                df['latitude'] = df['geometry'].apply(lambda g: g.centroid.xy[1][0])

                df['id'] = df['longitude'].astype(str) + ':' + df['latitude'].astype(str)
                print(f'\033[93m        Amount of buildings in S2 partition {len(df)}')

                buildings_in_tiles = []
                buildings_out_tiles = []

                buildings_found = 0
                buildings_out_tiles_found = 0

                # first step fast filteing all buildings that located in polygon grid tiles
                for in_poly_tile in tqdm(inside_polygon_grid, total=len(inside_polygon_grid),
                                         desc='Filtering buildings inside polygon'):

                    min_lon, min_lat, max_lon, max_lat = in_poly_tile.bounds
                    # Filter buildings within current tile
                    buildings_in_tile = df[
                        (df.longitude >= min_lon) &
                        (df.longitude <= max_lon) &
                        (df.latitude >= min_lat) &
                        (df.latitude <= max_lat)
                        ]

                    if len(buildings_in_tile) > 0:
                        buildings_in_tiles.append(buildings_in_tile)
                        buildings_found += len(buildings_in_tile)

                # first step fast filteing all buildings that located out polygon grid tiles
                for off_poly_tile in tqdm(outside_polygon_grid, total=len(outside_polygon_grid),
                                          desc='Filtering buildings outside polygon'):

                    min_lon, min_lat, max_lon, max_lat = off_poly_tile.bounds
                    # Filter buildings within current tile
                    buildings_out_tile = df[
                        (df.longitude >= min_lon) &
                        (df.longitude <= max_lon) &
                        (df.latitude >= min_lat) &
                        (df.latitude <= max_lat)
                        ]

                    if len(buildings_out_tile) > 0:
                        buildings_out_tiles.append(buildings_out_tile)
                        buildings_out_tiles_found += len(buildings_out_tile)

                # Concatenate dataframes containing buildins within and outside tiles

                try:
                    buildings_in_tiles_df = pd.concat(buildings_in_tiles)
                except:
                    buildings_in_tiles_df = pd.DataFrame()

                try:
                    buildings_out_tiles_df = pd.concat(buildings_out_tiles)
                except:
                    buildings_out_tiles_df = pd.DataFrame()

                # check in out buildings of polygon
                if (len(buildings_in_tiles_df) != 0) and (len(buildings_out_tiles_df) != 0):

                    try:
                        print(
                            f'\033[93m        Buildings found in polygon tiles {len(buildings_in_tiles_df)} out tiles {len(buildings_out_tiles_df)} remaining {len(df) - len(buildings_in_tiles_df) - len(buildings_out_tiles_df)}')

                        if len(buildings_in_tiles_df) > 0:
                            remaining_buildings_df = df[~df['id'].isin(buildings_in_tiles_df['id'])]

                            if len(buildings_out_tiles_df) > 0:
                                remaining_buildings_df = remaining_buildings_df[
                                    ~remaining_buildings_df['id'].isin(buildings_out_tiles_df['id'])]

                        if (len(buildings_in_tiles_df) == 0) and (len(buildings_out_tiles_df) > 0):
                            remaining_buildings_df = df[~df['id'].isin(buildings_out_tiles_df['id'])]

                        print(f'\033[93m        Remaining amount check {len(remaining_buildings_df)}')

                        result_df = pd.DataFrame()

                        if len(remaining_buildings_df) > 0:
                            remaining_buildings_df['buildings_in_polygon'] = [
                                region_polygon.contains(shapely.Point(row.longitude, row.latitude)) for row in
                                tqdm(remaining_buildings_df.itertuples(), total=len(remaining_buildings_df),
                                     desc='Filtering buildings')]

                            remaining_buildings_df = remaining_buildings_df[
                                remaining_buildings_df.buildings_in_polygon == True]

                            if len(buildings_in_tiles_df) > 0:
                                if len(remaining_buildings_df) > 0:
                                    result_df = pd.concat([buildings_in_tiles_df, remaining_buildings_df])
                                else:
                                    result_df = buildings_in_tiles_df
                            else:
                                if len(remaining_buildings_df) > 0:
                                    result_df = remaining_buildings_df
                                else:
                                    result_df = pd.DataFrame()
                        else:
                            if len(buildings_in_tiles_df) > 0:
                                result_df = buildings_in_tiles_df
                            else:
                                result_df = pd.DataFrame()

                        if len(result_df) > 0:
                            print(f'\033[93m        Saved amount {len(result_df)}')
                            
                            result_df.to_parquet(
                                f"{filtered_parquets_folder}/{region_name.replace(' ', '_')}_pidx_{pidx}_S2_id_{S2_id}_buildings.parquet")
                        # buildings_inside_polygon.append(result_df)

                        print("s")
                    except Exception as e:
                        print(e)
            except Exception as e:
                print(e)

Processing region Patan
    processing pidx 0
        processing file_name India_4132897083042562048.parquet idx 0 of 2
        Amount of buildings in S2 partition 1612243


Filtering buildings outside polygon: 100%|██████████| 11080/11080 [00:42<00:00, 262.86it/s]


        Buildings found in polygon tiles 1850 out tiles 14603 remaining 1595790
        Remaining amount check 1595790


Filtering buildings: 100%|██████████| 1595790/1595790 [00:27<00:00, 57376.74it/s]


        Saved amount 2293
s
        processing file_name India_4133460032995983360.parquet idx 1 of 2
        Amount of buildings in S2 partition 4587154


Filtering buildings outside polygon: 100%|██████████| 11080/11080 [02:03<00:00, 89.63it/s]


        Buildings found in polygon tiles 206786 out tiles 184328 remaining 4196040
        Remaining amount check 4196040


Filtering buildings: 100%|██████████| 4196040/4196040 [01:13<00:00, 57442.04it/s]


        Saved amount 211855
s


In [14]:
filtered_partitions = os.listdir(filtered_parquets_folder)
columns = ['bf_source', 'confidence', 'geometry', 'longitude', 'latitude', 'id']

for region_name in regions_polygons.keys():

    region_name = region_name.replace(' ', '_')
    try:
        filtered_partitions = [i for i in filtered_partitions if region_name in i]

        main_df = gpd.read_parquet(os.path.join(filtered_parquets_folder, filtered_partitions[0]), columns=columns)

        for i in filtered_partitions[1:]:
            
            parquet_path = os.path.join(filtered_parquets_folder, i)
            current_df = gpd.read_parquet(parquet_path, columns=columns)
            main_df = pd.concat([main_df, current_df])

        main_df = main_df.drop_duplicates(subset=['id'])
        
        filename = os.path.join(output_folder,f'{region_name}_buildings.parquet')
        
        main_df = calculations(main_df)
        main_df.to_parquet(filename)
        
        # upload to bucket
        try:
            res=cos_client.upload_file(Filename=filename, Bucket=config["VIDA_COUNTRIES_BUILDINGS"],Key=filename)
        except Exception as e:
            print(Exception, e)
        else:
            print(f'{filename} succesfully uploaded')
            
    except Exception as e:
        print(f'Region parquets concatenation error occurred: {e}')

/home/julian/notebooks/Patan_buildings.parquet succesfully uploaded


In [15]:
from datetime import datetime
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-10-02 08:33:42'